# Nettoyage de l’export mensuel — Jeanne

Objectif : contrôler l’export mensuel, comparer ses quantités
aux sommes horaires et reconstruire les mois complets.

Résultats obtenus sur les exports actuels :
- 31 mois sur 70 présentent des écarts.
- 68 mois complets conservés : février 2014 à septembre 2019.
- Janvier 2014 et octobre 2019 exclus de la prévision (incomplets).
- 13 pics signalés et conservés.

Les fichiers bruts ne sont jamais modifiés.

In [1]:
from pathlib import Path
import pandas as pd

# Retrouver la racine du projet
racine = Path.cwd()
if racine.name == "notebooks":
    racine = racine.parent

# Lire le fichier mensuel sans modifier l'original
mensuel = pd.read_csv(
    racine / "data" / "raw" / "Pharma_Ventes_Monthly.csv"
)

print("Dimensions :", mensuel.shape)
display(mensuel.head())

Dimensions : (70, 9)


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-31,132.02,99.750,154.600,915.030,360.0,50.0,115.0,52.2
1,2014-02-28,133.32,126.050,177.000,1001.900,347.0,31.0,122.0,36.2
2,2014-03-31,137.44,92.950,147.655,779.275,232.0,20.0,112.0,85.4
3,2014-04-30,119.44,91.475,132.900,718.500,216.0,18.0,97.0,77.7
4,2014-05-31,103.46,125.933,135.100,640.780,273.0,23.0,108.0,127.7


## 1. Contrôler les dates et les quantités

On recherche les dates invalides, les doublons, les mois absents
et les quantités manquantes ou négatives. Aucune ligne n’est supprimée.

In [2]:
groupes = [
    "M01AB", "M01AE", "N02BA", "N02BE",
    "N05B", "N05C", "R03", "R06"
]

mensuel["Date"] = pd.to_datetime(
    mensuel["datum"], format="%Y-%m-%d", errors="coerce"
)

mois = mensuel["Date"].dt.to_period("M")
calendrier = pd.period_range(mois.min(), mois.max(), freq="M")

controles = pd.Series({
    "Dates invalides": mensuel["Date"].isna().sum(),
    "Dates en double": mensuel["Date"].duplicated().sum(),
    "Mois en double": mois.duplicated().sum(),
    "Mois absents": len(calendrier.difference(mois)),
    "Quantités manquantes": mensuel[groupes].isna().sum().sum(),
    "Quantités négatives": (mensuel[groupes] < 0).sum().sum()
}, name="Nombre d'anomalies")

display(controles.to_frame())

,Nombre d'anomalies
Dates invalides,0
Dates en double,0
Mois en double,0
Mois absents,0
Quantités manquantes,0
Quantités négatives,0


## 2. Comparer les totaux mensuels aux sommes horaires

Les ventes horaires sont additionnées par mois et par groupe ATC.
Un écart supérieur à 0,000001 est signalé pour ignorer les imprécisions numériques.

In [3]:
horaire = pd.read_csv(
    racine / "data" / "raw" / "Pharma_Ventes_Hourly.csv"
)

horaire["Date"] = pd.to_datetime(
    horaire["datum"],
    format="%m/%d/%Y %H:%M",
    errors="raise"
)

assert not horaire["Date"].duplicated().any(), \
    "Dates horaires en double."
assert horaire[groupes].notna().all().all(), \
    "Quantités horaires manquantes."

horaire["Mois"] = horaire["Date"].dt.to_period("M")
recalcule = horaire.groupby("Mois")[groupes].sum()

original = mensuel.assign(Mois=mois).set_index("Mois")[groupes]

assert original.index.equals(recalcule.index), \
    "Les exports ne couvrent pas les mêmes mois."

ecarts = original - recalcule
anomalies = ecarts.abs() > 0.000001

print("Mois comparés :", len(ecarts))
print("Mois avec un écart :", anomalies.any(axis=1).sum())
print("Cases avec un écart :", anomalies.sum().sum())

# Afficher uniquement les écarts significatifs
display(ecarts.where(anomalies, 0).loc[anomalies.any(axis=1)])

Mois comparés : 70
Mois avec un écart : 31
Cases avec un écart : 45


,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
Mois,,,,,,,,
2014-01,0.000000,0.000000,0.000000,0.000000,0.000000,20.000,0.000000,0.0000
2014-04,0.000000,0.000000,5.750000,0.000000,0.000000,0.000,0.000000,0.0000
2014-08,10.000000,0.000000,0.000000,36.500000,0.000000,0.000,0.000000,0.0000
2014-09,0.000000,0.000000,10.000000,0.000000,0.000000,0.000,0.000000,0.0000
2014-10,53.555000,76.489688,51.062500,783.678125,39.937500,10.375,22.000000,27.8125
2015-03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,25.000000,0.0000
2015-06,0.000000,0.000000,0.000000,31.375000,0.000000,0.000,0.000000,0.0000
2015-11,10.000000,0.000000,0.000000,42.750000,0.000000,0.000,30.000000,0.0000
2016-03,0.000000,15.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.0000


## 3. Écarter les mois incomplets de la prévision

On compare le nombre d’heures présentes au nombre d’heures
du mois entier. Les mois incomplets restent dans les données brutes.

In [4]:
# Vérifier la continuité horaire entre les extrémités de l'export
heures_attendues = pd.date_range(
    horaire["Date"].min(),
    horaire["Date"].max(),
    freq="h"
)
heures_absentes = heures_attendues.difference(
    pd.DatetimeIndex(horaire["Date"])
)
assert len(heures_absentes) == 0, "Des heures sont absentes."

# Vérifier la couverture de chaque mois
couverture = pd.DataFrame({
    "Heures présentes": horaire.groupby("Mois").size(),
    "Heures attendues": recalcule.index.days_in_month * 24
})

couverture["Complet"] = (
    couverture["Heures présentes"] == couverture["Heures attendues"]
)

display(couverture.loc[~couverture["Complet"]])

mensuel_propre = recalcule.loc[couverture["Complet"]].copy()

print("Mois complets conservés :", len(mensuel_propre))
print("Premier mois :", mensuel_propre.index.min())
print("Dernier mois :", mensuel_propre.index.max())

,Heures présentes,Heures attendues,Complet
Mois,,,
2014-01,712,744,False
2019-10,188,744,False


Mois complets conservés : 68
Premier mois : 2014-02
Dernier mois : 2019-09


## 4. Signaler les pics sans les supprimer

Les quantités dépassant Q3 + 1,5 × (Q3 − Q1) sont signalées.
Ce seuil ne prouve pas une erreur : saisonnalité et changements
de demande peuvent expliquer ces valeurs.

In [5]:
q1 = mensuel_propre.quantile(0.25)
q3 = mensuel_propre.quantile(0.75)
seuils = q3 + 1.5 * (q3 - q1)

pics = mensuel_propre.gt(seuils)

detail_pics = pd.DataFrame([
    {
        "Mois": str(mois),
        "Groupe": groupe,
        "Quantité": mensuel_propre.loc[mois, groupe],
        "Seuil": seuils[groupe]
    }
    for mois in mensuel_propre.index
    for groupe in groupes
    if pics.loc[mois, groupe]
], columns=["Mois", "Groupe", "Quantité", "Seuil"])

print("Nombre de pics signalés :", len(detail_pics))
display(detail_pics.round(3))

Nombre de pics signalés : 13


,Mois,Groupe,Quantité,Seuil
0,2014-08,N05B,420.000,381.721
1,2014-09,N05B,399.000,381.721
2,2014-10,N05B,430.000,381.721
3,2014-11,N05B,489.000,381.721
4,2014-12,N05B,525.000,381.721
5,2015-01,N05B,478.000,381.721
6,2016-01,N02BA,187.000,185.688
7,2016-03,M01AE,165.589,160.269
8,2018-03,N05C,42.000,39.500
9,2018-12,R03,402.000,379.875


In [6]:
# Comparer les calculs du notebook au CSV produit par monthly.py
attendu = mensuel_propre.copy()
attendu.index = attendu.index.to_timestamp(how="start")
attendu.index.name = "Date"

enregistre = pd.read_csv(
    racine / "data" / "processed" / "ventes_mensuelles_propres.csv",
    parse_dates=["Date"]
).set_index("Date")

pd.testing.assert_frame_equal(
    enregistre,
    attendu,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-6,
    check_freq=False
)

print("Vérification réussie : notebook et script donnent les mêmes données.")

Vérification réussie : notebook et script donnent les mêmes données.
